# Find all mathing PMIDs
- had DRUG, but no DISEASE NER

In [88]:
import os
import pandas as pd
import ast
from tqdm import tqdm
import re
from collections import defaultdict


In [2]:
csv_directory = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions"  # Adjust as needed
drug_only_rows = []

# Helper function to check for DRUG but not DISEASE
def has_drug_not_disease(ner_string):
    try:
        if pd.isna(ner_string) or not isinstance(ner_string, str):
            return False
        entities = ast.literal_eval(ner_string.strip())
        has_drug = any(ent[2] == 'DRUG' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        has_disease = any(ent[2] == 'DISEASE' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        return has_drug and not has_disease
    except Exception as e:
        print(f"⚠️ Error parsing: {ner_string}\n{e}")
        return False

# Loop through all CSVs
csv_files = [f for f in os.listdir(csv_directory) if f.endswith(".csv")]
print(f"Processing {len(csv_files)} CSV files...\n")

for filename in tqdm(csv_files, desc="Scanning for DRUG-only rows"):
    filepath = os.path.join(csv_directory, filename)

    try:
        df = pd.read_csv(filepath)

        if 'ner_prediction_BioLinkBERT-base_normalized' not in df.columns:
            print("columnd with predictions not found")
            continue

        df['source_file'] = filename
        filtered = df[df['ner_prediction_BioLinkBERT-base_normalized'].apply(has_drug_not_disease)]

        if not filtered.empty:
            drug_only_rows.append(filtered)

    except Exception as e:
        tqdm.write(f"Error reading {filename}: {e}")

# Combine all DRUG-only rows
if drug_only_rows:
    result_df = pd.concat(drug_only_rows, ignore_index=True)
    print(f"\n✅ Done. Found {len(result_df)} DRUG-only rows.")
else:
    result_df = pd.DataFrame()
    print("\n⚠️ Done. No DRUG-only rows found.")

Processing 617 CSV files...



Scanning for DRUG-only rows: 100%|██████████| 617/617 [03:28<00:00,  2.96it/s]


✅ Done. Found 1038588 DRUG-only rows.


In [4]:
result_df.head()

,PMID,ner_prediction_BioLinkBERT-base_normalized,source_file
0,22274300,"[(21, 32, 'DRUG', 'fenofibrate'), (107, 126, '...",test_annotated_BioLinkBERT-base_tuples_2025032...
1,22274401,"[(15, 24, 'DRUG', 'vitamin a'), (299, 308, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
2,22274564,"[(1142, 1163, 'DRUG', 'pparα agonist wy14643')...",test_annotated_BioLinkBERT-base_tuples_2025032...
3,22274565,"[(0, 10, 'DRUG', 'estrogenic'), (369, 377, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
4,22274614,"[(86, 91, 'DRUG', 'miraculin -')]",test_annotated_BioLinkBERT-base_tuples_2025032...


In [5]:
result_df.to_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions/drug_no_disease.csv",index=False)

In [2]:
result_df = pd.read_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions/drug_no_disease.csv")

In [3]:
target_pmids = result_df['PMID'].astype(str).unique()
len(target_pmids)

1010433

# Get content (title/abstract)

In [14]:
new_folder = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large"
filtered_dfs = []

csv_files = [f for f in os.listdir(new_folder) if f.endswith(".csv")]
print(f"Scanning {len(csv_files)} files for matching PMIDs...\n")

for filename in tqdm(csv_files, desc="Filtering new data"):
    filepath = os.path.join(new_folder, filename)

    try:
        df = pd.read_csv(filepath)

        if 'PMID' not in df.columns:
            continue

        df['PMID'] = df['PMID'].astype(str)
        matches = df[df['PMID'].isin(target_pmids)].copy()

        if not matches.empty:
            matches['source_file'] = filename
            filtered_dfs.append(matches)

    except Exception as e:
        tqdm.write(f"Error in {filename}: {e}")

if filtered_dfs:
    combined_text_df = pd.concat(filtered_dfs, ignore_index=True)
    print(f"\n✅ Found {len(combined_text_df)} matching rows across new files.")
else:
    combined_text_df = pd.DataFrame()
    print("\n⚠️ No matching PMIDs found in the new files.")

Scanning 618 files for matching PMIDs...



Filtering new data: 100%|██████████| 618/618 [05:06<00:00,  2.02it/s]



✅ Found 2077176 matching rows across new files.


In [15]:
combined_text_df.head()

,PMID,Text,source_file,ner_prediction_BioLinkBERT-base_normalized
0,19118090,Modulation of PKCdelta signaling alters the sh...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
1,19118094,Ventromedial nucleus neurons are less sensitiv...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
2,19118099,Expression of rainbow trout glucose transporte...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
3,19118104,"OAG induces an additional PKC-, PI3K-, and Rac...",pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
4,19118163,VASP is involved in cAMP-mediated Rac 1 activa...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN


In [16]:
combined_text_df.to_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_no_disease_content.csv",index=False)

In [4]:
combined_text_df = pd.read_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_no_disease_content.csv")

/sctmp/sdonev/ipykernel_2257456/987797916.py:1: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_text_df = pd.read_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_no_disease_content.csv")


In [5]:
combined_text_df.head()

,PMID,Text,source_file,ner_prediction_BioLinkBERT-base_normalized
0,19118090,Modulation of PKCdelta signaling alters the sh...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
1,19118094,Ventromedial nucleus neurons are less sensitiv...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
2,19118099,Expression of rainbow trout glucose transporte...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
3,19118104,"OAG induces an additional PKC-, PI3K-, and Rac...",pubmed_filtered_animal_for_NER_chunk_523.csv,NaN
4,19118163,VASP is involved in cAMP-mediated Rac 1 activa...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN


# Run Model KW to Disease Map

In [6]:
text_df = combined_text_df.copy()

In [31]:

# Phrase mapping dictionary (example — extend as needed)
regex_disease_models = [

    # --- Multiple Sclerosis (model-specific, EAE only) ---
    (re.compile(r"""
            experimental\s+(autoimmune|allergic)\s+encephalomyelitis  |
            experimentally\s+demyelinated                            |
            \bEAE\b                                    
        """, re.I | re.X),
         "multiple sclerosis"),

    # --- Multiple Sclerosis (general mentions) ---
    (
        re.compile(r"\bmultiple\s+sclerosis\b", re.I),
        "multiple sclerosis"
    ),

    # --- PARKINSON’S DISEASE – NEUROTOXIC MODELS (only when used in model context) ---
    (
        re.compile(r"""
            \b(mptp|6[-\s]?ohda|maneb)
            (?:[-–\s]+(induced|lesion|model))\b
        """, re.I | re.X),
        "parkinson's disease"
    ),
    
    # --- General mentions: Parkinson's disease, parkinsonism, parkinsonian ---
    (
        re.compile(r"""
            \b
            parkinson(?:['’]s\s+disease|ism|ian)
            \b
        """, re.I | re.X),
        "parkinson's disease"
    ),


   # --- Alzheimer’s disease models ---
    (
        re.compile(r"""
            \b(
                pdapp |
                tg2576 |
                app23 |
                j20 |
                (app[\-/]?)ps1[0-9a-z]* |
                appswe[\-/]?ps1d?e9 |
                tgswdi |
                tgf344[-\s]?ad |
                3x[tT]g(-ad)? |
                5xfad |
                app[e‚e]693[δdΔ]?-?tg |
                app[-\s]?knock[-\s]?in[-\s]?nl[-\s]?g[-\s]?f |
                mcgill[-\s]?r[-\s]?thy1[-\s]?app
            )\b
        """, re.X),
        "alzheimer's disease"
    ),

    # --- General Alzheimer mentions ---
    (
        re.compile(r"""
            \b
            alzheimer(?:['’]s)?       # Alzheimer's / Alzheimer
            (?:\s+disease)?           # Optional 'disease'
            \b
        """, re.I | re.X),
        "alzheimer's disease"
    )
]


# ──────────────────────────────────────────────────────────────────────────────
# 2.  EXTRACTION FUNCTION – returns a list of tuples exactly like:
#       (start, end, 'DISEASE', raw_phrase)
#       (start, end, 'DISEASE', normalised_name)
# ──────────────────────────────────────────────────────────────────────────────
def extract_neuro_disease_models(text: str):
    """Return list of NER-style tuples for well-established neuro-disease models."""
    text = str(text)
    results = []

    for rgx, disease_name in regex_disease_models:
        for m in rgx.finditer(text):
            start, end = m.start(), m.end()
            raw_phrase = m.group(0)
            # raw match
            results.append((start, end, 'DISEASE_model', raw_phrase))
            # normalised mapping
            results.append((start, end, 'DISEASE', disease_name))

    return results

In [32]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.  APPLY TO YOUR DATAFRAME
# ──────────────────────────────────────────────────────────────────────────────
tqdm.pandas(desc="Extracting disease models")
text_df['disease_mentions_from_model'] = text_df['Text'].progress_apply(extract_neuro_disease_models)


Extracting disease models: 100%|██████████| 2077176/2077176 [03:16<00:00, 10562.83it/s] 


In [33]:
text_df.head()

,PMID,Text,source_file,ner_prediction_BioLinkBERT-base_normalized,disease_mentions_from_model
0,19118090,Modulation of PKCdelta signaling alters the sh...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,[]
1,19118094,Ventromedial nucleus neurons are less sensitiv...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,[]
2,19118099,Expression of rainbow trout glucose transporte...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,[]
3,19118104,"OAG induces an additional PKC-, PI3K-, and Rac...",pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,[]
4,19118163,VASP is involved in cAMP-mediated Rac 1 activa...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,[]


In [34]:
pmid_file = "./missing_ner_filter_CONTENT.csv"  # <-- Change this
pmid_df = pd.read_csv(pmid_file, sep='|')  # Use sep based on file format
target_pmids = set(pmid_df['pmid'].astype(str))

In [36]:
df_disease_found_pmids = set(df_disease_found['PMID'].astype(str))

In [40]:
df_disease_found.head()

,PMID,Text,source_file,ner_prediction_BioLinkBERT-base_normalized,disease_mentions_from_model
190,10377370,Rat strain differences in the ability to disru...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,"[(1782, 1796, DISEASE_model, 6-OHDA-induced), ..."
209,10385681,Characterization and implications of estrogeni...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,"[(419, 438, DISEASE_model, Parkinson's disease..."
368,9329348,Cortisol reduces hippocampal glucose metabolis...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,"[(78, 97, DISEASE_model, Alzheimer's disease),..."
685,35783620,Bio-Evaluation of [99m]Tc-Labeled Homodimeric ...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,"[(1508, 1519, DISEASE_model, Alzheimer's), (15..."
1322,17204747,Inhibition of acrolein-induced apoptosis by th...,pubmed_filtered_animal_for_NER_chunk_523.csv,NaN,"[(344, 363, DISEASE_model, Alzheimer's disease..."


In [37]:
len(target_pmids & df_disease_found_pmids)

47

In [38]:
mentions_df = (
    text_df[['PMID', 'disease_mentions_from_model']]
    .explode('disease_mentions_from_model')
    .dropna(subset=['disease_mentions_from_model'])
)
mentions_df['label'] = mentions_df['disease_mentions_from_model'].apply(lambda t: t[2])
mentions_df['value'] = mentions_df['disease_mentions_from_model'].apply(lambda t: t[3])

# Create mapping of PMID to normalized diseases
norm_diseases = (
    mentions_df[mentions_df['label'] == 'DISEASE']
    .groupby('PMID')['value']
    .unique()
    .to_dict()
)

# Filter only raw model phrases
model_mentions_df = mentions_df[mentions_df['label'] == 'DISEASE_model'].copy()

# Add normalized disease from PMID mapping
model_mentions_df['diseases_from_same_doc'] = model_mentions_df['PMID'].map(norm_diseases)

model_mentions_df = model_mentions_df.explode('diseases_from_same_doc')

model_summary = (
    model_mentions_df
    .groupby(['diseases_from_same_doc', 'value'])['PMID']
    .nunique()
    .reset_index(name='unique_PMID_count')
    .sort_values(by=['diseases_from_same_doc', 'unique_PMID_count'], ascending=[True, False])
)
pmid_samples = (
    model_mentions_df
    .groupby(['diseases_from_same_doc', 'value'])['PMID']
    .apply(lambda pmids: list(pd.Series(pmids.unique()).sample(
        n=min(3, len(pmids.unique())),
        random_state=42,
        replace=False
    )) if len(pmids.unique()) > 0 else [])
    .reset_index(name='PMID_sample')
)

# Merge with summary table
model_summary = model_summary.merge(pmid_samples, on=['diseases_from_same_doc', 'value'])
model_summary

,diseases_from_same_doc,value,unique_PMID_count,PMID_sample
0,alzheimer's disease,Alzheimer's disease,1053,"[15175078, 17406977, 20041722]"
1,alzheimer's disease,Alzheimer's,288,"[32535316, 35617665, 39655335]"
2,alzheimer's disease,Alzheimer,168,"[10389174, 7675877, 12514509]"
3,alzheimer's disease,Parkinson's disease,128,"[31170199, 35921496, 32073100]"
4,alzheimer's disease,Alzheimer disease,99,"[15975079, 11277574, 24381170]"
...,...,...,...,...
66,parkinson's disease,MPTP-lesion,1,[9602028]
67,parkinson's disease,MPTP-model,1,[8635534]
68,parkinson's disease,Maneb induced,1,[30529917]
69,parkinson's disease,Parkinson’s disease,1,[37736880]


In [39]:
model_summary.to_csv("./mapped_disease_from_animal_model_v2.csv")

In [43]:
df_disease_found.shape

(4622, 5)

In [41]:
result_df.head()

,PMID,ner_prediction_BioLinkBERT-base_normalized,source_file
0,22274300,"[(21, 32, 'DRUG', 'fenofibrate'), (107, 126, '...",test_annotated_BioLinkBERT-base_tuples_2025032...
1,22274401,"[(15, 24, 'DRUG', 'vitamin a'), (299, 308, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
2,22274564,"[(1142, 1163, 'DRUG', 'pparα agonist wy14643')...",test_annotated_BioLinkBERT-base_tuples_2025032...
3,22274565,"[(0, 10, 'DRUG', 'estrogenic'), (369, 377, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
4,22274614,"[(86, 91, 'DRUG', 'miraculin -')]",test_annotated_BioLinkBERT-base_tuples_2025032...


In [108]:
df_disease_found_with_drug = df_disease_found[['PMID','disease_mentions_from_model']].merge(result_df[['PMID','ner_prediction_BioLinkBERT-base_normalized']], on="PMID", how='left')
df_disease_found_with_drug = df_disease_found_with_drug.drop_duplicates(subset="PMID")

In [109]:
df_disease_found_with_drug.shape

(4491, 3)

In [110]:
df_disease_found_with_drug.to_csv("./out_model_to_disease_map/df_disease_found_with_drug.csv",index=False)

In [111]:
df_disease_found_with_drug.head()

,PMID,disease_mentions_from_model,ner_prediction_BioLinkBERT-base_normalized
0,10377370,"[(1782, 1796, DISEASE_model, 6-OHDA-induced), ...","[(420, 456, 'DRUG', 'hydroxy - 2 - ( di - n - ..."
1,10385681,"[(419, 438, DISEASE_model, Parkinson's disease...","[(602, 610, 'DRUG', 'estrogen'), (721, 738, 'D..."
2,9329348,"[(78, 97, DISEASE_model, Alzheimer's disease),...","[(0, 8, 'DRUG', 'cortisol'), (101, 116, 'DRUG'..."
3,35783620,"[(1508, 1519, DISEASE_model, Alzheimer's), (15...","[(46, 54, 'DRUG', 'chalcone'), (259, 267, 'DRU..."
4,17204747,"[(344, 363, DISEASE_model, Alzheimer's disease...","[(60, 76, 'DRUG', 'n - acetylcysteine'), (417,..."


In [112]:
import ast
import pandas as pd

def safe_convert_to_list(value):
    """Safely convert value to list"""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return []
    return []

# Dictionary to track each disease value and PMIDs
disease_details = defaultdict(lambda: {'count': 0, 'pmids': set()})

def merge_disease_mentions_detailed_tracking(row):
    global disease_details
    
    # Convert both columns to lists safely
    existing_predictions = safe_convert_to_list(row['ner_prediction_BioLinkBERT-base_normalized'])
    disease_mentions = safe_convert_to_list(row['disease_mentions_from_model'])
    pmid = row['PMID']
    
    # Add disease mentions where third element equals 'DISEASE'
    for mention in disease_mentions:
        if len(mention) >= 3 and 'DISEASE' == str(mention[2]):
            existing_predictions.append(mention)
            # Track the disease value (4th element of tuple)
            if len(mention) >= 4:
                disease_value = mention[3]  # Assuming disease name is 4th element
                disease_details[disease_value]['count'] += 1
                disease_details[disease_value]['pmids'].add(pmid)
    
    return existing_predictions

# Apply the function
df_disease_found_with_drug['ner_prediction_BioLinkBERT-base_normalized'] = df_disease_found_with_drug.apply(merge_disease_mentions_detailed_tracking, axis=1)

# Create a summary DataFrame for easier analysis
summary_data = []
for disease, info in disease_details.items():
    summary_data.append({
        'disease': disease,
        'total_mentions': info['count'],
        'unique_pmids': len(info['pmids']),
        'pmids_list': sorted(list(info['pmids']))
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('unique_pmids', ascending=False)

# Print results
print("DISEASE VALUES AND PMID COUNTS:")
print("=" * 80)
for _, row in summary_df.iterrows():
    print(f"Disease: {row['disease']}")
    print(f"  Total mentions: {row['total_mentions']}")
    print(f"  Unique PMIDs: {row['unique_pmids']}")
    print("-" * 60)

print(f"\nSUMMARY STATISTICS:")
print(f"Total unique diseases: {len(summary_df)}")
print(f"Total disease mentions added: {summary_df['total_mentions'].sum()}")
print(f"Total unique PMIDs with diseases: {len(set().union(*summary_df['pmids_list']))}")

# Show diseases sorted by number of PMIDs
print(f"\nTOP DISEASES BY NUMBER OF PMIDs:")
print("=" * 50)
print(summary_df[['disease', 'unique_pmids', 'total_mentions']].to_string(index=False))


DISEASE VALUES AND PMID COUNTS:
Disease: parkinson's disease
  Total mentions: 2492
  Unique PMIDs: 1631
------------------------------------------------------------
Disease: alzheimer's disease
  Total mentions: 1790
  Unique PMIDs: 1611
------------------------------------------------------------
Disease: multiple sclerosis
  Total mentions: 5748
  Unique PMIDs: 1413
------------------------------------------------------------

SUMMARY STATISTICS:
Total unique diseases: 3
Total disease mentions added: 10030
Total unique PMIDs with diseases: 4491

TOP DISEASES BY NUMBER OF PMIDs:
            disease  unique_pmids  total_mentions
parkinson's disease          1631            2492
alzheimer's disease          1611            1790
 multiple sclerosis          1413            5748


In [113]:
df_disease_found_with_drug.iloc[3]['ner_prediction_BioLinkBERT-base_normalized']

[(46, 54, 'DRUG', 'chalcone'),
 (259, 267, 'DRUG', 'chalcone'),
 (814, 820, 'DRUG', 'dt ch )'),
 (1508, 1519, 'DISEASE', "alzheimer's disease")]

In [114]:
df_disease_found_with_drug[['PMID','ner_prediction_BioLinkBERT-base_normalized']].to_csv("./out_model_to_disease_map/from_model_disease_with_drug.csv",index=False)

In [115]:
df_disease_found_with_drug.PMID.nunique()

4491